# Guardião Gate — experimento offline reproduzível

Este notebook organiza preparação, execução e análise dos **56 casos sintéticos**: 36 centrais e vinte variantes normal/maliciosa. Cada execução completa cria uma nova rodada e preserva as anteriores.

**Pergunta central:** em que medida o controlador impede ações não autorizadas, preservando tarefas permitidas e rastreabilidade? Aqui avaliamos os controles com **sinais fixos e ferramentas simuladas**. Nenhum modelo, serviço de internet ou ação externa real será utilizado.

Referências: [protocolo](../docs/planejamento/protocolo-experimental.md), [contrato JEV](../docs/especificacao/contrato-jev.md) e [avaliação offline](../docs/especificacao/avaliacao-offline-completa.md).

Para reproduzir: salve o arquivo, escolha o ambiente com as dependências fixadas, **reinicie o kernel e execute todas as células em ordem**. Não é necessário restaurar os corpora completos em `data/raw/`.


## 1. Localizar o projeto e carregar os componentes

Funciona com o diretório de trabalho na raiz ou em `notebooks/`. Não há caminho absoluto específico deste computador. Opcionalmente, `GUARDIAO_PROJECT_ROOT` pode indicar a raiz de outra cópia completa do projeto.


In [ ]:
from pathlib import Path
import os
import sys

_candidates = ([Path(os.environ["GUARDIAO_PROJECT_ROOT"]).expanduser()]
               if os.environ.get("GUARDIAO_PROJECT_ROOT")
               else [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in _candidates
             if (p / "data/cases/manifesto.json").is_file()
             and (p / "src/guardiao_gate").is_dir()
             and (p / "config/notebook-offline-1.0.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Projeto não encontrado. Abra a cópia completa na raiz ou em notebooks/.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from IPython.display import HTML, Markdown, display
from guardiao_gate import PilotData
from guardiao_gate.notebook import (
    analyze_run, html_table, load_notebook_config, persist_analysis,
    prepare_notebook, run_from_notebook,
)
print("Projeto:", ROOT)
print("Interpretador:", sys.executable)


## 2. Configuração explícita

O modo implementado é `offline`. As regras, entradas e resultados esperados ficam nos arquivos externos; não são reescritos nas células. Ativar modelo nesta versão gera erro antes de criar uma rodada.

As versões das dependências são conferidas por padrão. A referência é Python 3.12.8/macOS, com pacotes em `requirements-notebook.txt`. A instalação inicial pode precisar de internet; a execução do experimento não precisa.


In [ ]:
MODO = "offline"
USAR_MODELO = False

PARAMETERS = load_notebook_config(ROOT)
PARAMETERS["mode"] = MODO
PARAMETERS["model_enabled"] = USAR_MODELO

PREPARATION = prepare_notebook(ROOT, PARAMETERS)
display(Markdown(f"**Modo:** {MODO}. **Python observado:** {PREPARATION['python_version']}."))
display(HTML(html_table(PREPARATION["dependencies"])))


## 3. Conferir entradas e integridade

O carregador verifica IDs, hashes, separação das entradas e correspondência dos casos com seus oráculos. Arquivo ausente ou alterado impede a execução. Esta conferência **não executa o experimento** nem valida juridicamente as referências.

D07 permanece restrito. A inspeção abaixo mostra apenas metadados; não carrega seu conteúdo no contexto comum.


In [ ]:
DATA = PilotData(ROOT)
INTEGRITY = DATA.validate()
_inventory = [{"material": name, "quantidade": value} for name, value in [
    ("Documentos-base", INTEGRITY["base_documents"]),
    ("Variantes pareadas", INTEGRITY["paired_documents"]),
    ("Estados", INTEGRITY["states"]),
    ("Casos centrais", INTEGRITY["central_cases"]),
    ("Casos pareados", INTEGRITY["paired_cases"]),
]]
display(HTML(html_table(_inventory)))
print("Integridade:", INTEGRITY["data_integrity"])
print("Contrato normalizado:", INTEGRITY["signals_contract"])


## 4. Estado confirmado, ferramentas e responsabilidades

A fase e as permissões vêm da configuração confiável. Texto de documento não altera estado, mandato ou aprovação. Os efeitos operacionais de EMAIL, PRECO e MIGRAR prevalecem sobre seus nomes ou sinais favoráveis do classificador.

`—` ou `null` significa ausência/desconhecido, conforme o campo; **não significa zero**.


In [ ]:
POLICY = DATA._json("data/cases/politica.json")
STATES = DATA._json("data/cases/estados.json")["states"]
display(HTML(html_table(POLICY["tools"])))
display(HTML(html_table(STATES, ["scenario_id", "phase", "release", "conditions", "closed", "autonomy_level"])))
print("Destinatário fixo do mock:", POLICY["email_recipient"])
print("Revisão humana de domínio:", POLICY["human_domain_review"])


## 5. Executar a rodada completa

O executor aplica todos os casos e preserva divergências e falhas. O controlador decide antes de o avaliador consultar as respostas esperadas. As aprovações são eventos do executor confiável, não mensagens encontradas nos documentos.

A rodada cria `results/suite-offline-.../`, contendo entradas de referência, versões, hashes, decisões, aprovações, invocações, recibos e relatório. As sondagens proibidas são propostas explícitas do avaliador; não serão contadas como tentativas espontâneas de um modelo.


In [ ]:
RUN_DIR, METRICS = run_from_notebook(ROOT, PREPARATION)
print("Rodada:", METRICS["run_id"])
print("Registros preservados em:", RUN_DIR)
print("Casos coletados:", METRICS["case_count"])
print("Chamadas a modelos:", METRICS["model_calls"])
print("Ações externas reais:", METRICS["real_external_actions"])


## 6. Ler os resultados preservados

A análise usa os arquivos da rodada, conferidos pelo manifesto, e não o estado interno de um controlador ainda na memória. Casos com falhas técnicas previstas só coincidem quando produzem o estado e a contagem corretos.


In [ ]:
ANALYSIS = analyze_run(RUN_DIR)
display(HTML(html_table(ANALYSIS["groups"])))
display(HTML(html_table(ANALYSIS["controls"])))
_compliance = METRICS["approval_compliance"]
print("Invocações supervisionadas com aprovação válida:",
      f"{_compliance['valid_approval_invocations']}/{_compliance['supervised_invocations']}")
print("Escopo dessa contagem:", METRICS["approval_compliance_scope"])
print("Invocações atuais:", METRICS["target_invocations"])
print("Invocações com histórico:", METRICS["total_invocations_including_history"])


## 7. Conferir os 36 casos centrais

A tabela compara o observado à referência por caso. T16 separa a tentativa corrente do histórico confirmado. T33 contém duas requisições, com uma execução e uma duplicata; a ordem do vencedor pode variar.


In [ ]:
display(HTML(html_table(ANALYSIS["central"], [
    "case_id", "decision_initial", "result_status", "dispatch_invocations",
    "expected_invocations", "history_invocations", "observed_match",
])))


## 8. Conferir os vinte casos pareados

Cada variante B/M tem uma leitura legítima e resposta interna por extração e critérios fixos. A coluna de utilidade é restrita à rubrica offline; não demonstra a capacidade de resumo ou revisão de um modelo.

Há 26 sondagens dentro dos vinte casos: P03 tem duas propostas e P09 três, em cada variante. Elas não aumentam o conjunto para 82 casos.


In [ ]:
display(HTML(html_table(ANALYSIS["paired_groups"])))
display(HTML(html_table(ANALYSIS["paired"], [
    "case_id", "result_status", "dispatch_invocations", "expected_invocations",
    "utility_fixed_rubric", "observed_match",
])))
print("Sondagens injetadas:", METRICS["evaluator_probes"])
print("Invocações resultantes das sondagens:", METRICS["probe_invocations"])


## 9. Inspecionar falhas e efeitos desconhecidos

T31/T32 devem preservar `null` para efeito confirmado, exigir reconciliação e impedir reenvio automático. T30 registra indisponibilidade da trilha normal e um diagnóstico separado de emergência. Passar nesses casos significa tratar a falha corretamente, sem alegar sucesso confirmado.


In [ ]:
display(HTML(html_table(ANALYSIS["critical"])))
display(HTML(html_table(ANALYSIS["statuses"])))
print("Casos com efeito desconhecido:", METRICS["unknown_effect_cases"])
print("Casos com auditoria indisponível:", METRICS["audit_unavailable_cases"])


## 10. Tornar divergências visíveis

A coleta completa já foi preservada antes desta análise. Uma divergência não apaga os demais casos e não é removida do denominador. Quando houver problemas, o notebook os apresenta e ainda exporta a análise.


In [ ]:
if ANALYSIS["divergences"]:
    display(HTML(html_table([
        {"caso": case["case_id"],
         "criterios_nao_atendidos": [key for key, ok in case["evaluation"]["checks"].items() if not ok],
         "erro_executor": case.get("execution_error")}
        for case in ANALYSIS["divergences"]
    ])))
else:
    display(Markdown("Nenhuma divergência registrada nesta rodada."))
print("Critérios técnicos locais atendidos:", METRICS["technical_readiness"])


## 11. Etapas com modelo e processo B0/B1

| Frente | Situação nesta entrega |
|---|---|
| Controle offline | Executado nesta rodada |
| Agente e JEV real da Typesafe | Não integrados; item 6 |
| Comparação técnica com/sem controlador | Não executada |
| Geração B0 e comparação B0/B1 | Configuração disponível; execução e comparação no item 7 |
| Utilidade de modelo e sucesso espontâneo de ataques | Não aplicáveis nesta rodada |

Não se calculam ganhos de tempo ou qualidade a partir desta suíte. As versões normais/maliciosas são casos conhecidos de desenvolvimento, sem alegação de generalização.


In [ ]:
display(HTML(html_table([
    {"indicador": "Utilidade de modelo", "valor": METRICS["model_utility"]},
    {"indicador": "Sucesso espontâneo de ataques", "valor": METRICS["model_attack_success_rate"]},
    {"indicador": "Chamadas a modelos", "valor": METRICS["model_calls"]},
])))
display(Markdown("**Limites da interpretação:**\n\n" + "\n".join("- " + text for text in ANALYSIS["limits"])))


## 12. Preservar a análise e localizar a rodada

A análise é salva em uma subpasta própria, com manifesto ligado à rodada. A reexecução completa gera outra pasta, sem sobrescrever esta. Se quiser executar novamente, reinicie o kernel e rode todas as células.

O comando `python3 scripts/executar_notebook.py` também salva uma cópia executada do notebook e seu registro de execução em kernel limpo. O arquivo-fonte versionado continua sem saídas antigas.


In [ ]:
ANALYSIS_DIR = persist_analysis(RUN_DIR, ANALYSIS)
_relative_run = RUN_DIR.relative_to(ROOT).as_posix()
display(Markdown(
    f"**Rodada:** `{METRICS['run_id']}`\n\n"
    f"- [Relatório dos 56 casos](../{_relative_run}/relatorio.md)\n"
    f"- [Casos observados](../{_relative_run}/casos-observados.csv)\n"
    f"- [Divergências](../{_relative_run}/divergencias.json)\n"
    f"- [Manifesto da rodada](../{_relative_run}/manifesto.json)\n"
    f"- [Análise do notebook](../{_relative_run}/analise-notebook/analise.md)"
))
print("Análise preservada em:", ANALYSIS_DIR)
print("Resultado local:", "critérios atendidos" if METRICS["technical_readiness"] else "divergências a investigar")
